# Tutorial: 3D Image Classification  

> SynapseMNIST3D dataset demo


In [ ]:
#| default_exp tutorial_classification_3D

### Setup imports

In [ ]:
from bioMONAI.data import *
from bioMONAI.transforms import *
from bioMONAI.core import *
from bioMONAI.core import Path
#from bioMONAI.data import get_image_files, get_target, RandomSplitter
from bioMONAI.losses import *
from bioMONAI.metrics import *
from bioMONAI.datasets import download_medmnist
from bioMONAI.visualize import show_images_grid, mosaic_image_3d


#from monai.utils import set_determinism
#from monai.transforms import ScaleIntensity

#set_determinism(0)

In [ ]:
device = get_device()
print(device)

### Download and store the dataset

In [ ]:
data_flag = 'synapsemnist3d'
data_path = Path('../_data/medmnist_data/')

info = download_medmnist(data_flag, data_path, download_only=True)

data_path = data_path/'synapsemnist3d'
train_path = data_path/'train'
val_path = data_path/'val'
test_path = data_path/'test'

### Create Dataloader

In [ ]:
batch_size = 4

data = BioDataLoaders.class_from_folder(
    data_path,
    train='train',
    valid='val',
    vocab=info['label'],
    item_tfms=[ScaleIntensity(), RandRot90(prob=0.5), Resize(32)],
    batch_tfms=None,
    img_cls=BioImageStack,
    bs=batch_size,
    )

# print length of training and validation datasets
print('train images:', len(data.train_ds.items), '\nvalidation images:', len(data.valid_ds.items))

In [ ]:
#im, label = data.train_ds[0]
#print(type(im), im.shape, label, label.shape)

In [ ]:
# visualization (ERROR PARA DATOS 3D)
data.show_batch()

### Load and train a 3D model

In [ ]:
from monai.networks.nets import DenseNet121, HighResNet
from fastai.vision.all import BalancedAccuracy, CrossEntropyLossFlat, xresnet34
# from torch.nn import CrossEntropyLoss

#model = HighResNet
model = DenseNet121(spatial_dims=3, in_channels=1, out_channels=1)
#model = xresnet34(n_out=2)

loss = CrossEntropyLossFlat()
metric = BalancedAccuracy

trainer = fastTrainer(data, model, loss_fn=loss, metrics=metric, show_summary=True)

In [ ]:
trainer.fit(1)

In [ ]:
#trainer.show_results(cmap='gray')

In [ ]:
# trainer.save('tmp-model')

### Test data 
Evaluate the performance of the selected model on unseen data.
It’s important to not touch this data until you have fine tuned your model to get an unbiased evaluation!